Adapted from https://github.com/dargueso/EHF/

In [1]:
import sys
sys.path.append('.')

import os
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

import netCDF4 as nc
import xarray as xr
import numpy as np
import glob as glob
import pandas as pd

import pdb
from itertools import groupby
from Constants import const
from pathlib import Path
import datetime as dt

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client

In [2]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest/"
write_path = '/scratch/ng72/ms5578/hw_files/'

In [3]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 7
Total threads: 14,Total memory: 125.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37329,Workers: 7
Dashboard: http://127.0.0.1:8787/status,Total threads: 14
Started: Just now,Total memory: 125.19 GiB
Comm: tcp://127.0.0.1:46559,Total threads: 2
Dashboard: http://127.0.0.1:38555/status,Memory: 17.88 GiB
Nanny: tcp://127.0.0.1:33523,


2025-05-03 14:20:30,012 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a2c41edcf8b9e0dd2a8d266cd869912d initialized by task ('rechunk-merge-rechunk-transfer-51499933c329897ab14096fa2e2aa291', 0, 0, 0, 8, 0, 0) executed on worker tcp://127.0.0.1:38509
2025-05-03 14:20:33,903 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a2c41edcf8b9e0dd2a8d266cd869912d deactivated due to stimulus 'task-finished-1746246033.901603'
2025-05-03 14:22:48,468 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a2c41edcf8b9e0dd2a8d266cd869912d initialized by task ('rechunk-merge-rechunk-transfer-51499933c329897ab14096fa2e2aa291', 0, 0, 0, 5, 0, 0) executed on worker tcp://127.0.0.1:44425
2025-05-03 14:22:51,233 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a2c41edcf8b9e0dd2a8d266cd869912d deactivated due to stimulus 'task-finished-1746246171.2288103'
2025-05-03 14:24:59,602 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a2c41edcf8b9e0dd2a8d266cd8699

In [4]:
def calc_percentile(tave, nyears, thres_file=None, method="NF13", nwindow=15):

    """Function to calculate the percentile that indentifies hot days
    tave: mean daily temperature calcualted from tmax and tmin
    nyears: number of years in the analysed period
    thres_file: file that contains previously calculated percentiles
    method: now two methods are supported depending on how the percentiles are calculated 'NF13' and 'PA13'
    nwindow: number of days for the window used to calculate percentiles in PA13 method
    ---
    output: pct_calc
    """
    if method == "NF13":

        if thres_file == None:
            print("No thresholds file provided, we will calculate them")

            if not isinstance(tave, np.ma.core.MaskedArray):
                pct_calc = np.nanpercentile(tave, 95, axis=0)

            else:
                pct_calc = np.ones(tave.shape[1:], float) * const.missingval
                for i in range(tave.shape[1]):
                    for j in range(tave.shape[2]):
                        aux = tave[:, i, j]
                        if len(aux[~aux.mask].data) != 0:
                            pct_calc[i, j] = np.nanpercentile(
                                aux[~aux.mask].data, 95, axis=0
                            )
        else:
            print("Percentiles are retrieved from the thfile provided")
            pct_file = nc.Dataset(thres_file, "r")
            pct_calc = pct_file.variables["PRCTILE95"][:].astype("float")

    elif method == "PA13":

        if thres_file == None:
            # No percentile file is provided and thus they are calculated from the given data
            print("Percentiles are calculated because no thfile is provided")
            windowrange = np.zeros((365,), dtype=bool)
            windowrange[: int(np.ceil(nwindow / 2))] = True
            windowrange[-int(np.floor(nwindow / 2)) :] = True
            if np.sum(windowrange) != nwindow:
                raise SystemExit(0)
            windowrange = np.tile(windowrange, nyears)
            pct_calc = np.ones((365,) + tave.shape[1:], float) * const.missingval

            if not isinstance(tave, np.ma.core.MaskedArray):
                for d in range(365):
                    pct_calc[d, :, :] = np.percentile(
                        tave[windowrange == True, :, :], 90, axis=0
                    )
                    windowrange = np.roll(windowrange, 1)

            else:
                for i in range(tave.shape[1]):
                    for j in range(tave.shape[2]):
                        for d in range(365):
                            aux = tave[windowrange == True, :, :]
                            if len(aux[~aux.mask].data) != 0:
                                pct_calc[d, :, :] = np.percentile(
                                    aux[~aux.mask].data, 90, axis=0
                                )
                            windowrange = np.roll(windowrange, 1)

        else:
            print("Percentiles are retrieved from the thfile provided")
            # A percentile file is provided and it contains a PRCTILE90 variable
            pct_file = nc.Dataset(thres_file, "r")
            pct_calc = pct_file.variables["PRCTILE90"][:].astype("float")

    else:
        raise ValueError("Method not supported: Choose between NF13 or PA13")

    return pct_calc


In [5]:
def calc_spell(series):

    if isinstance(series, np.ma.core.MaskedArray):
        if np.any(series.mask == True):
            series[series.mask] = -99

    srun = np.zeros(series.shape)
    srun[1:] = np.diff(series, axis=0)
    srun[srun == 99] = -1
    srun[srun == 100] = 1
    srun[0] = -1
    if isinstance(series, np.ma.core.MaskedArray):
        L = (series.data).tolist()
    else:
        L = (series).tolist()
    groups_hw = []

    for k, g in groupby(L):
        if k == 1:
            b = list(g)
            groups_hw.append(sum(b))

    spell_hw = np.zeros((len(series),), dtype=int)
    if np.any(srun == 1):
        spell_hw[srun == 1] = np.asarray(groups_hw)

    ## Keep only spells equal or larger than 3 days

    spell_hw[spell_hw < 3] = 0
    return spell_hw

In [6]:
def tave_tm_bnds(mask,dates,tave):

    """Function to calculate Excess Heat Factor (EHF) heatwaves from tave calcualted as (tmax+tmin)/2."""
    if mask == None:
        mask = np.ones(tave.shape[1:], int)

    # PERFORM SOME CHECKS
    ## This is explicitly checked to preserve compatibility across versions
    if (bsyear == None) or (beyear == None):
        sys.exit(
            "ERROR: you didn't provide base period years to compute_EHF function, please revise"
        )
        
    years_all = np.asarray([dates[i].year for i in range(len(dates))])
    months_all = np.asarray([dates[i].month for i in range(len(dates))])
    days_all = np.asarray([dates[i].day for i in range(len(dates))])

    # If using PA13, leap days need to be removed

    if method == "PA13":

        dates = dates[((months_all == 2) & (days_all == 29)) == False]
        years = np.asarray([dates[i].year for i in range(len(dates))])
        months = np.asarray([dates[i].month for i in range(len(dates))])
        days = np.asarray([dates[i].day for i in range(len(dates))])

        tave = tave[((months_all == 2) & (days_all == 29)) == False, :, :]

    else:

        years = np.asarray([dates[i].year for i in range(len(dates))])
        months = np.asarray([dates[i].month for i in range(len(dates))])
        days = np.asarray([dates[i].day for i in range(len(dates))])

    # Specify when the year start
    # It is important to define seasons (e.g. Souther Hemisphere, month_starty should be in winter)
    new_years = years.copy()
    new_years[months < month_starty] -= 1

    syear = np.min(years)
    eyear = np.max(years)
    nyears = eyear - syear
    nbyears = beyear - bsyear

    shift_pct = np.argmax(new_years == syear)

    ndays = tave.shape[0]
    nlat = tave.shape[1]
    nlon = tave.shape[2]

    return tave, nbyears, years

In [7]:
def calc_EHF():
    tave_3days = np.zeros(tave.shape, dtype=float)
    tave_3days = tave.rolling(time=3).mean().fillna(0)

    if EHFaccl == True:
        tave_30days = np.zeros(tave.shape, dtype=float)
        tave_30days = tave.rolling(time=30).mean().fillna(0)

    ###############################################
    ###############################################
    ### CALCULATING EHIsig and EHIaccl (if required)

    if method == "PA13":
        EHIsig = np.zeros(tave.shape, dtype=float)
        for t in range(ndays):
            EHIsig[t, :, :] = tave_3days[t, :, :] - pct[(t) % 365, :, :]
    else:
        EHIsig = tave_3days - pct

    if EHFaccl == True:
        EHIaccl = tave_3days - tave_30days

    ###############################################
    ###############################################
    ## CALCULATING EHF and EHF_Exceed

    if EHFaccl == True:
        EHF = np.maximum(1, EHIaccl) * EHIsig
    else:
        EHF = EHIsig
        
    EHF = EHF.where(EHF > 0, 0)
    EHF_exceed = EHF.where(EHF <= 0, 1)

    if EHFaccl == True:
        del tave_30days, EHIaccl, EHIsig
    
    return EHF, EHF_exceed, tave_3days

In [8]:
def zero_days():
    ###### ZEROING DAYS NOT BELONGING TO SUMMER (SH: NOV,DEC,JAN,FEB,MAR; NH: MAY,JUN,JUL,AUG,SEP)
    ###### Originally used only in PA13 method
    if season == "summer_sh":
        EHF_exceed[(months >= 4) & (months <= 10), :, :] = False
        years[(months >= 4) & (months <= 10)] = -99

        ## For heat wave timing purposes
        shift_start_year = (
            dt.datetime(syear, 11, 0o1) - dt.datetime(syear, 0o7, 0o1)
        ).days

    elif season == "summer_nh":
        EHF_exceed[(months >= 10) | (months <= 4), :, :] = False
        years[(months >= 10) | (months <= 4)] = -99
        shift_start_year = 0

    elif season == "yearly":
        shift_start_year = 0
    else:
        raise ValueError(
            "Season not supported: Choose between summer_sh, summer_nh or yearly"
        )
    return EHF_exceed

In [9]:
def compute_heatwave_metrics(EHF_exceed_1D, EHF_1D, TMP3D_1D, mask):
    ndays = EHF_1D.shape[0]
    spell = calc_spell(EHF_exceed_1D)

    # Initialize outputs
    avg = np.full_like(EHF_1D, const.missingval)
    peak = np.full_like(EHF_1D, const.missingval)
    tmp_peak = np.full_like(EHF_1D, const.missingval)
    tmp_avg = np.full_like(EHF_1D, const.missingval)
    ehf_flag = np.zeros_like(EHF_1D)

    if bool(mask):  # Ensure works with Dask and NumPy scalars
        t = 0
        while t < ndays:
            if spell[t] != 0:
                span = spell[t]
                avg[t] = np.mean(EHF_1D[t : t + span])
                peak[t] = np.max(EHF_1D[t : t + span])
                tmp_peak[t] = np.max(TMP3D_1D[t : t + span])
                tmp_avg[t] = np.mean(TMP3D_1D[t : t + span])
                ehf_flag[t : t + span] = EHF_exceed_1D[t : t + span]
                t += span
            else:
                t += 1

    return avg, peak, tmp_peak, tmp_avg, ehf_flag

In [10]:
def open_files(files):
    fin = xr.open_mfdataset(files, concat_dim ='time', combine='nested', 
                            parallel=True, data_vars='minimal',coords='minimal', 
                            drop_variables = "time_bnds",chunks="auto")

    tave = fin.tas.chunk(time=-1,lat=-1,lon="350Mb")
    dt64 = fin.time.values
    dates = pd.to_datetime(dt64)
    
    fin.close()

    return tave, dates

In [11]:
thres_file="/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/t95_baseline.nc"
bsyear=1979
beyear=2000

month_starty=7
EHFaccl=True
method="NF13"
mask = xr.open_dataarray("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/land_sea_mask.nc").astype(bool)
season="yearly"

In [12]:
sdate, edate ='20180701', '20200630'

fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
fpaths = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fpaths])

tave, dates = open_files(fpaths)
tave, nbyears, years = tave_tm_bnds(True,dates,tave)
pct = calc_percentile(
    tave[(years >= bsyear) & (years <= beyear), :, :],
    nbyears,
    thres_file,
    method=method,
    nwindow=15,
)
EHF, EHF_exceed, tave_3days = calc_EHF()
heatwave_EHF_avg, heatwave_EHF_peak, heatwave_TMP3D_peak, heatwave_TMP3D_ave, heatwave_EHF = xr.apply_ufunc(
                                                                                                        compute_heatwave_metrics,
                                                                                                        EHF_exceed,
                                                                                                        EHF,
                                                                                                        tave_3days,
                                                                                                        mask,
                                                                                                        input_core_dims=[['time'], ['time'], ['time'],[]],
                                                                                                        output_core_dims=[['time'], ['time'], ['time'], ['time'], ['time']],
                                                                                                        vectorize=True,
                                                                                                        dask='parallelized',
                                                                                                        output_dtypes=[float, float, float, float, float]
                                                                                                        )

del heatwave_TMP3D_peak, heatwave_TMP3D_ave

HW_EHF = tave.to_dataset().assign(EHF_flag=heatwave_EHF, HW_EHF_avg=heatwave_EHF_avg, HW_EHF_peak = heatwave_EHF_peak)

start_julys = pd.date_range(start=dates[0], end=dates[-1], freq='12MS')

encoding = {v: {"zlib": True, "complevel": 4, "shuffle": True} for v in HW_EHF.data_vars}

for start in start_julys:
    end = start + pd.DateOffset(months=12) - pd.DateOffset(days=1)
    chunk = HW_EHF.sel(time=slice(start, end)).compute()
    if chunk.time.size > 0:
        chunk.to_netcdf(f"{write_path}HW_EHF_{start.year}_{(start + pd.DateOffset(months=11)).year}.nc",
                       encoding=encoding,
                       compute=False,
                       engine='netcdf4')
        del chunk


Percentiles are retrieved from the thfile provided


In [13]:
ehf_vals = heatwave_EHF.compute()
ehf_vals

<xarray.DataArray (lat: 646, lon: 1082, time: 731)> Size: 4GB
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
...
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]])
Coordinates:
  * time     (time) datetime64[ns] 6kB 2018-07-01T12:00:00 ... 2020-06-30T12:...
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
    height   float64 8B 1.5
    crs      int32 4B 0

In [14]:
np.unique(ehf_vals)

array([0., 1.])